# Part 1: Data Loading & Cleaning

In [1]:
# Import neccessary libraries
import pandas as pd
import numpy as np

In [2]:
# Read the csvs into dataframes
df1 = pd.read_csv("options_data_earnings/sp500_options_pit_2016-2020.csv")
df2 = pd.read_csv("options_data_earnings/sp500_options_pit_2020-2025.csv")

In [3]:
# Check batch 1 & 2
print(f"Batch 1: {len(df1):,} rows | years {sorted(df1['constituent_year'].unique())}")
print(f"Batch 2: {len(df2):,} rows | years {sorted(df2['constituent_year'].unique())}")

Batch 1: 13,894 rows | years [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
Batch 2: 63,380 rows | years [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [4]:
df = pd.concat([df1, df2], ignore_index=True)
print(f"Combined raw      : {len(df):,} rows")

Combined raw      : 77,274 rows


In [5]:
df["earnings_date"] = pd.to_datetime(df["earnings_date"])
df["entry_date"]    = pd.to_datetime(df["entry_date"])
df["exit_date"]     = pd.to_datetime(df["exit_date"])

In [6]:
KEY    = ["ticker","earnings_date","entry_offset","exit_offset"]
before = len(df)
df = (df.sort_values("entry_straddle_mid", ascending=False)
        .drop_duplicates(subset=KEY, keep="first")
        .reset_index(drop=True))
print(f"After dedup       : {len(df):,} rows (removed {before - len(df):,})")

df = df[~((df["entry_offset"]==0) & (df["exit_offset"]==0))].copy()
print(f"After T0/T0 excl. : {len(df):,} rows")

print(f"\nEntry offsets: {sorted(df['entry_offset'].unique())}")
print(f"Exit offsets : {sorted(df['exit_offset'].unique())}")
print(f"Tickers      : {df['ticker'].nunique():,}")
print(f"Events       : {df['earnings_date'].nunique():,}")

After dedup       : 77,274 rows (removed 0)
After T0/T0 excl. : 69,599 rows

Entry offsets: [np.int64(-3), np.int64(-2), np.int64(-1), np.int64(0)]
Exit offsets : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
Tickers      : 551
Events       : 1,536


# Part 2: Configuration Analysis

In [7]:
# Import Neccessary Libraries
import yfinance as yf
import time
import plotly.graph_objects as go

In [8]:
tickers = df["ticker"].unique().tolist()
sector_map = {}

for i, t in enumerate(tickers):
    try:
        info = yf.Ticker(t).info
        sector_map[t] = info.get("sector", "Unknown")
    except Exception:
        sector_map[t] = "Unknown"
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(tickers)} fetched...")
    time.sleep(0.3)

df["sector"] = df["ticker"].map(sector_map)

print(f"\nSector mapping complete:")
print(df["sector"].value_counts().to_string())
print(f"\nUnknown: {(df['sector']=='Unknown').sum()}")

  50/551 fetched...
  100/551 fetched...
  150/551 fetched...
  200/551 fetched...
  250/551 fetched...
  300/551 fetched...
  350/551 fetched...
  400/551 fetched...
  450/551 fetched...
  500/551 fetched...
  550/551 fetched...

Sector mapping complete:
sector
Technology                15428
Consumer Cyclical         11821
Industrials               10233
Healthcare                 8963
Financial Services         6461
Communication Services     4212
Consumer Defensive         3724
Basic Materials            2801
Energy                     2485
Utilities                  1635
Real Estate                1322
Unknown                     514

Unknown: 514


In [10]:
# Set Up NUSA Theme
BG    = "#ffffff"
PANEL = "#f8f8f8"
RED   = "#e8133a"
GREEN = "#2db82d"
GREY  = "#555555"
DG    = "#dddddd"
FONT  = "Garamond, 'EB Garamond', Georgia, serif"

e_offs = [-3, -2, -1, 0]
x_offs = [0, 1, 2, 3]

SECTORS = sorted(df[df["sector"] != "Unknown"]["sector"].unique())
print(f"Sectors available: {SECTORS}")

Sectors available: ['Basic Materials', 'Communication Services', 'Consumer Cyclical', 'Consumer Defensive', 'Energy', 'Financial Services', 'Healthcare', 'Industrials', 'Real Estate', 'Technology', 'Utilities']


In [11]:
# Define the plot function
def plot_sector(sector=None):
    sub   = df if sector is None else df[df["sector"] == sector]
    label = "All Sectors" if sector is None else sector

    st = (sub.groupby(["entry_offset","exit_offset"])
             .agg(
                 n        = ("pnl_pct","count"),
                 avg_ret  = ("pnl_pct","mean"),
                 win_rate = ("pnl_pct", lambda x: (x>0).mean()),
             )
             .reset_index())
    idx_s = st.set_index(["entry_offset","exit_offset"])

    z, txt = [], []
    for e in e_offs:
        rz, rt = [], []
        for x in x_offs:
            if e == 0 and x == 0:
                rz.append(None); rt.append("T0/T0\nexcl.")
            elif (e, x) in idx_s.index:
                v  = idx_s.loc[(e,x), "avg_ret"]
                wr = idx_s.loc[(e,x), "win_rate"]
                n  = int(idx_s.loc[(e,x), "n"])
                rz.append(v)
                rt.append(f"{v:.2%}\nWR {wr:.0%}  n={n}")
            else:
                rz.append(None); rt.append("")
        z.append(rz); txt.append(rt)

    fig = go.Figure(go.Heatmap(
        z=z,
        x=[f"Exit T{x:+d}" for x in x_offs],
        y=[f"Entry T{e:+d}" for e in e_offs],
        text=txt, texttemplate="%{text}",
        textfont=dict(family=FONT, size=12, color="#111111"),
        colorscale=[[0,RED],[0.45,"#f5a0aa"],[0.5,"#ffffff"],
                    [0.55,"#a0e6a0"],[1,GREEN]],
        zmid=0, showscale=True,
        colorbar=dict(tickfont=dict(family=FONT, color=GREY, size=11),
                      outlinecolor=DG, outlinewidth=1),
        hoverongaps=False,
        hovertemplate="Entry %{y} / Exit %{x}<br>%{text}<extra></extra>",
    ))
    fig.update_layout(
        title=dict(
            text=f"S&P 500 Earnings Straddle  ·  Avg Return per Combo  ({label})",
            font=dict(family=FONT, size=19, color="#111111"),
            x=0.03, xanchor="left"),
        paper_bgcolor=BG, plot_bgcolor=PANEL,
        font=dict(family=FONT, color="#111111", size=13),
        height=440, margin=dict(l=65, r=40, t=70, b=40),
        hoverlabel=dict(bgcolor="#ffffff", bordercolor=DG,
                        font=dict(family=FONT, color="#111111", size=13)),
    )
    fig.update_xaxes(side="top", tickfont=dict(family=FONT, color="#111111", size=13),
                     gridcolor=DG, linecolor=DG)
    fig.update_yaxes(autorange="reversed",
                     tickfont=dict(family=FONT, color="#111111", size=13),
                     gridcolor=DG, linecolor=DG)
    fig.show()

In [12]:
plot_sector("Technology")

# Part 3: Long-Short Straddle Strategy on Tech

In [16]:
# Define training and test sets
train = df[(df["sector"] == "Technology") & (df["constituent_year"] <= 2021)].copy()
test  = df[(df["sector"] == "Technology") & (df["constituent_year"] >= 2022)].copy()

print(f"Training set : {len(train):,} rows | years {sorted(train['constituent_year'].unique())}")
print(f"Test set     : {len(test):,}  rows | years {sorted(test['constituent_year'].unique())}")

Training set : 6,082 rows | years [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Test set     : 8,007  rows | years [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [27]:
# Set Filters & Configuration
LONG_AVG_RET  =  0.01
LONG_MED_RET  =  0.00
LONG_MIN_N    =  20

SHORT_AVG_RET = -0.01
SHORT_MED_RET =  0.00
SHORT_MIN_N   =  20

CAPITAL_PER_TRADE = 1_000
PORTFOLIO_SIZE    = 10_000 # For ROI calculation

In [28]:
# Identify signals on Training Data
combo_stats = (train.groupby(["ticker","entry_offset","exit_offset"])
                    .agg(
                        n        = ("pnl_pct","count"),
                        avg_ret  = ("pnl_pct","mean"),
                        med_ret  = ("pnl_pct","median"),
                        win_rate = ("pnl_pct", lambda x: (x>0).mean()),
                        std      = ("pnl_pct","std"),
                    )
                    .reset_index())

long_signals = combo_stats[
    (combo_stats["n"]       > LONG_MIN_N)   &
    (combo_stats["avg_ret"] > LONG_AVG_RET) &
    (combo_stats["med_ret"] > LONG_MED_RET)
].copy()
long_signals["side"] = "LONG"

short_signals = combo_stats[
    (combo_stats["n"]       > SHORT_MIN_N)   &
    (combo_stats["avg_ret"] < SHORT_AVG_RET) &
    (combo_stats["med_ret"] < SHORT_MED_RET)
].copy()
short_signals["side"] = "SHORT"

print(f"Signals identified from training period (2016-2021):")
print(f"  Long  signals : {len(long_signals)}")
print(f"  Short signals : {len(short_signals)}")

print(f"\nLong signals:")
print(long_signals[["ticker","entry_offset","exit_offset","n","avg_ret","med_ret","win_rate"]]
      .sort_values("avg_ret", ascending=False)
      .to_string(index=False, formatters={
          "avg_ret":  "{:.2%}".format,
          "med_ret":  "{:.2%}".format,
          "win_rate": "{:.0%}".format,
      }))

print(f"\nShort signals:")
print(short_signals[["ticker","entry_offset","exit_offset","n","avg_ret","med_ret","win_rate"]]
      .sort_values("avg_ret")
      .to_string(index=False, formatters={
          "avg_ret":  "{:.2%}".format,
          "med_ret":  "{:.2%}".format,
          "win_rate": "{:.0%}".format,
      }))

Signals identified from training period (2016-2021):
  Long  signals : 4
  Short signals : 34

Long signals:
ticker  entry_offset  exit_offset  n avg_ret med_ret win_rate
  AAPL            -1            1 21   6.44%   4.11%      52%
  AAPL            -2            0 21   2.40%   0.16%      52%
  AAPL            -1            0 21   2.22%   1.08%      57%
  ADBE            -1            0 21   1.60%   0.92%      57%

Short signals:
ticker  entry_offset  exit_offset  n avg_ret med_ret win_rate
  ADBE            -3            2 21 -23.25% -35.59%      24%
  ADBE            -3            3 21 -22.49% -26.02%      19%
  ADBE            -3            1 21 -19.21% -27.24%      14%
  ADBE            -2            2 22 -16.56% -35.69%      27%
  ADBE            -2            3 22 -16.19% -24.07%      32%
    MU            -1            2 22 -11.93% -24.16%      27%
  ADBE            -2            1 22 -11.93% -21.56%      32%
  NVDA            -2            1 22 -11.59% -22.33%      27%
    MU 

In [23]:
# Apply signals to the Test Data
long_keys  = set(zip(long_signals["ticker"],
                     long_signals["entry_offset"],
                     long_signals["exit_offset"]))
short_keys = set(zip(short_signals["ticker"],
                     short_signals["entry_offset"],
                     short_signals["exit_offset"]))

test_signals = test.copy()
test_signals["signal"] = None
test_signals.loc[test_signals.apply(
    lambda r: (r["ticker"],r["entry_offset"],r["exit_offset"]) in long_keys, axis=1),
    "signal"] = "LONG"
test_signals.loc[test_signals.apply(
    lambda r: (r["ticker"],r["entry_offset"],r["exit_offset"]) in short_keys, axis=1),
    "signal"] = "SHORT"

test_signals = test_signals[test_signals["signal"].notna()].copy()
test_signals["strategy_pnl"] = test_signals.apply(
    lambda r: r["pnl_pct"] if r["signal"]=="LONG" else -r["pnl_pct"], axis=1)

print(f"2022-2025 test trades: {len(test_signals)}")
print(f"  Long  trades : {(test_signals['signal']=='LONG').sum()}")
print(f"  Short trades : {(test_signals['signal']=='SHORT').sum()}")


2022-2025 test trades: 457
  Long  trades : 63
  Short trades : 394


In [29]:
# Evaluate the out of sample performance
def perf(returns, label):
    r   = pd.Series(returns).dropna()
    inv = len(r) * CAPITAL_PER_TRADE
    print(f"\n  {label}")
    print(f"    Trades          : {len(r)}")
    print(f"    Capital deployed: ${inv:,.0f}")
    print(f"    Win rate        : {(r>0).mean():.1%}")
    print(f"    Avg return      : {r.mean():.2%}")
    print(f"    Median ret      : {r.median():.2%}")
    print(f"    Total PnL       : ${r.sum()*CAPITAL_PER_TRADE:,.2f}")
    print(f"    ROI             : {r.sum()*CAPITAL_PER_TRADE/PORTFOLIO_SIZE:.2%}")
    print(f"    Best trade      : ${r.max()*CAPITAL_PER_TRADE:,.2f}")
    print(f"    Worst trade     : ${r.min()*CAPITAL_PER_TRADE:,.2f}")

print(f"\nOut-of-Sample Performance (2022-2025):")
perf(test_signals["strategy_pnl"], "Combined Long + Short")
perf(test_signals[test_signals["signal"]=="LONG"]["strategy_pnl"],  "Long only")
perf(test_signals[test_signals["signal"]=="SHORT"]["strategy_pnl"], "Short only")


Out-of-Sample Performance (2022-2025):

  Combined Long + Short
    Trades          : 457
    Capital deployed: $457,000
    Win rate        : 60.8%
    Avg return      : 4.97%
    Median ret      : 5.26%
    Total PnL       : $22,733.75
    ROI             : 227.34%
    Best trade      : $1,012.38
    Worst trade     : $-1,942.35

  Long only
    Trades          : 63
    Capital deployed: $63,000
    Win rate        : 41.3%
    Avg return      : 0.55%
    Median ret      : -1.22%
    Total PnL       : $343.52
    ROI             : 3.44%
    Best trade      : $1,012.38
    Worst trade     : $-419.20

  Short only
    Trades          : 394
    Capital deployed: $394,000
    Win rate        : 64.0%
    Avg return      : 5.68%
    Median ret      : 8.39%
    Total PnL       : $22,390.23
    ROI             : 223.90%
    Best trade      : $695.97
    Worst trade     : $-1,942.35


In [30]:
# Perfornmance by Ticker
print(f"\nPerformance by Ticker:")
ticker_perf = (test_signals.groupby(["ticker","signal"])
               .agg(
                   n        = ("strategy_pnl","count"),
                   avg_ret  = ("strategy_pnl","mean"),
                   win_rate = ("strategy_pnl", lambda x: (x>0).mean()),
                   total    = ("strategy_pnl","sum"),
               )
               .round(4))
print(ticker_perf.to_string(formatters={
    "avg_ret":  "{:.2%}".format,
    "win_rate": "{:.0%}".format,
    "total":    "{:.2f}".format,
}))


Performance by Ticker:
                 n avg_ret win_rate total
ticker signal                            
AAPL   LONG     48  -1.71%      38% -0.82
ADBE   LONG     15   7.75%      53%  1.16
       SHORT   165   5.55%      64%  9.15
ADSK   SHORT    13   0.57%      31%  0.07
CRM    SHORT    28  -0.64%      50% -0.18
MU     SHORT    91   7.24%      67%  6.59
NVDA   SHORT    76   4.27%      70%  3.25
QRVO   SHORT    21  16.72%      67%  3.51


In [32]:
# Train vs Test
print(f"\nTraining (2016-2021) vs Test (2022-2025) — Signal Combos:")
print(f"{'Ticker':<8} {'E':>3} {'X':>3} {'Side':<6} "
      f"{'Train Avg':>10} {'Train WR':>9} {'Train N':>8} "
      f"{'Test Avg':>9} {'Test WR':>8} {'Test N':>7}")

all_signals = pd.concat([long_signals, short_signals])
for _, row in all_signals.sort_values(["side","ticker"]).iterrows():
    t, e, x, side = row["ticker"], int(row["entry_offset"]), int(row["exit_offset"]), row["side"]
    test_sub = test_signals[(test_signals["ticker"]==t) &
                             (test_signals["entry_offset"]==e) &
                             (test_signals["exit_offset"]==x)]
    if len(test_sub) == 0:
        test_avg, test_wr, test_n = float("nan"), float("nan"), 0
    else:
        pnl      = test_sub["strategy_pnl"]
        test_avg = pnl.mean(); test_wr = (pnl>0).mean(); test_n = len(pnl)

    print(f"{t:<8} {e:>+3} {x:>+3} {side:<6} "
          f"{row['avg_ret']:>10.2%} {row['win_rate']:>9.0%} {int(row['n']):>8} "
          f"{test_avg:>9.2%} {test_wr:>8.0%} {test_n:>7}")


Training (2016-2021) vs Test (2022-2025) — Signal Combos:
Ticker     E   X Side    Train Avg  Train WR  Train N  Test Avg  Test WR  Test N
AAPL      -2  +0 LONG        2.40%       52%       21     2.61%      56%      16
AAPL      -1  +0 LONG        2.22%       57%       21    -0.25%      38%      16
AAPL      -1  +1 LONG        6.44%       52%       21    -7.48%      19%      16
ADBE      -1  +0 LONG        1.60%       57%       21     7.75%      53%      15
ADBE      -3  +0 SHORT      -4.83%       29%       21     3.65%      77%      13
ADBE      -3  +1 SHORT     -19.21%       14%       21     6.85%      69%      13
ADBE      -3  +2 SHORT     -23.25%       24%       21    10.98%      64%      11
ADBE      -3  +3 SHORT     -22.49%       19%       21    18.25%      75%      12
ADBE      -2  +0 SHORT      -1.86%       36%       22    -3.49%      73%      15
ADBE      -2  +1 SHORT     -11.93%       32%       22    -3.00%      60%      15
ADBE      -2  +2 SHORT     -16.56%       27%      

In [35]:
# Overall PnL evaluation
tech_df = df[df["sector"] == "Technology"].copy()
tech_df["signal"] = None
tech_df.loc[tech_df.apply(
    lambda r: (r["ticker"],r["entry_offset"],r["exit_offset"]) in long_keys, axis=1),
    "signal"] = "LONG"
tech_df.loc[tech_df.apply(
    lambda r: (r["ticker"],r["entry_offset"],r["exit_offset"]) in short_keys, axis=1),
    "signal"] = "SHORT"

strat = tech_df[tech_df["signal"].notna()].copy()
strat["strategy_pnl"] = strat.apply(
    lambda r: r["pnl_pct"] if r["signal"]=="LONG" else -r["pnl_pct"], axis=1)
strat["period"]     = strat["constituent_year"].apply(
    lambda y: "train" if y <= 2021 else "test")
strat["dollar_pnl"] = strat["strategy_pnl"] * CAPITAL_PER_TRADE
strat = strat.sort_values("exit_date").reset_index(drop=True)

strat["cum_dollar_pnl"]  = strat.groupby("period")["dollar_pnl"].cumsum()
strat["overall_cum_pnl"] = strat["dollar_pnl"].cumsum()
strat["total_invested"]  = (strat.index + 1) * CAPITAL_PER_TRADE
strat["peak"]            = strat["overall_cum_pnl"].cummax()
strat["drawdown"]        = strat["overall_cum_pnl"] - strat["peak"]

print(f"\nOverall Strategy Performance  (${CAPITAL_PER_TRADE:,} per trade)")
for period, grp in strat.groupby("period"):
    r   = grp["dollar_pnl"]
    inv = len(grp) * CAPITAL_PER_TRADE
    print(f"\n  {period.upper()} ({grp['constituent_year'].min()}-{grp['constituent_year'].max()})")
    print(f"    Trades          : {len(r)}")
    print(f"    Capital deployed: ${inv:,.0f}")
    print(f"    Total PnL       : ${r.sum():,.2f}")
    print(f"    ROI             : {r.sum()/inv:.2%}")
    print(f"    Win rate        : {(r>0).mean():.1%}")
    print(f"    Avg per trade   : ${r.mean():,.2f}")
    print(f"    Best trade      : ${r.max():,.2f}")
    print(f"    Worst trade     : ${r.min():,.2f}")

r   = strat["dollar_pnl"]
inv = len(strat) * CAPITAL_PER_TRADE
print(f"\n  FULL PERIOD (2016-2025)")
print(f"    Trades          : {len(r)}")
print(f"    Capital deployed: ${inv:,.0f}")
print(f"    Total PnL       : ${r.sum():,.2f}")
print(f"    ROI             : {r.sum()/inv:.2%}")
print(f"    Win rate        : {(r>0).mean():.1%}")
print(f"    Avg per trade   : ${r.mean():,.2f}")
print(f"    Max Drawdown    : ${strat['drawdown'].min():,.2f}")
print(f"    Best trade      : ${r.max():,.2f}")
print(f"    Worst trade     : ${r.min():,.2f}")


Overall Strategy Performance  ($1,000 per trade)

  TEST (2022-2025)
    Trades          : 457
    Capital deployed: $457,000
    Total PnL       : $22,733.75
    ROI             : 4.97%
    Win rate        : 60.8%
    Avg per trade   : $49.75
    Best trade      : $1,012.38
    Worst trade     : $-1,942.35

  TRAIN (2016-2021)
    Trades          : 835
    Capital deployed: $835,000
    Total PnL       : $64,372.37
    ROI             : 7.71%
    Win rate        : 68.6%
    Avg per trade   : $77.09
    Best trade      : $846.80
    Worst trade     : $-2,645.80

  FULL PERIOD (2016-2025)
    Trades          : 1292
    Capital deployed: $1,292,000
    Total PnL       : $87,106.12
    ROI             : 6.74%
    Win rate        : 65.9%
    Avg per trade   : $67.42
    Max Drawdown    : $-20,202.48
    Best trade      : $1,012.38
    Worst trade     : $-2,645.80


In [42]:
fig = go.Figure()
for period, col, name in [("train", "#2E74B5", "Training 2016-2021"),
                           ("test",  GREEN,     "Test 2022-2025")]:
    grp = strat[strat["period"]==period]
    fig.add_trace(go.Scatter(
        x=grp["exit_date"], y=grp["cum_dollar_pnl"],
        mode="lines", name=name,
        line=dict(color=col, width=2),
        hovertemplate="%{x|%b %Y}<br>Cumulative PnL: $%{y:,.0f}<extra></extra>",
    ))

fig.add_vline(x=pd.Timestamp("2022-01-01").timestamp()*1000,
              line_color=DG, line_dash="dash",
              annotation_text="2022 test begins",
              annotation_font=dict(family=FONT, size=11, color=GREY))
fig.add_hline(y=0, line_color=DG, line_dash="dot")

fig.update_layout(
    title=dict(text=f"Technology Earnings Straddle  ·  Cumulative PnL  (${CAPITAL_PER_TRADE:,} per trade)",
               font=dict(family=FONT, size=19, color="#111111"),
               x=0.03, xanchor="left"),
    paper_bgcolor=BG, plot_bgcolor=PANEL,
    font=dict(family=FONT, color="#111111", size=13),
    height=460, margin=dict(l=80, r=40, t=70, b=60),
    legend=dict(bgcolor=BG, bordercolor=DG, borderwidth=1,
                font=dict(family=FONT, size=12)),
    hoverlabel=dict(bgcolor="#ffffff", bordercolor=DG,
                    font=dict(family=FONT, color="#111111", size=13)),
    xaxis=dict(gridcolor=DG, linecolor=DG,
               tickfont=dict(family=FONT, color="#333333")),
    yaxis=dict(title="Cumulative PnL ($)",
               tickprefix="$", tickformat=",.0f",
               gridcolor=DG, linecolor=DG, zerolinecolor=DG,
               tickfont=dict(family=FONT, color="#333333"),
               title_font=dict(family=FONT, color="#333333", size=12)),
)
fig.show()